In [2]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import roc_auc_score
from tensorflow.keras.layers import Dense, Activation, BatchNormalization, LSTM, Masking, Input, GRU, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1
from tensorflow.keras import regularizers
from sklearn.utils import shuffle
from qkeras import *
import qkeras
from tensorflow.keras.models import load_model
from qkeras.utils import model_quantize
from qkeras.utils import model_save_quantized_weights

2025-11-03 13:52:31.377684: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-03 13:52:31.379648: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-03 13:52:31.407883: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-03 13:52:31.407914: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-03 13:52:31.409213: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

## LSTM

In [3]:
def lstmmodel(max_len, n_var, rec_units, ndense=[10], l1_reg=0,
              l2_reg=0, rec_act='sigmoid', extra_lab='none', rec_kernel_init='VarianceScaling',
             dense_kernel_init='lecun_uniform', domask=False):
    
    hidden = x_in = Input(shape=(max_len, n_var,))
    hidden = LSTM(units=rec_units,
                  recurrent_activation = rec_act,
                  kernel_initializer = rec_kernel_init, 
                  name = 'lstm1')(hidden)
    hidden = Dense(50, kernel_initializer=dense_kernel_init, name='dense_0' )(hidden)
    hidden = Activation('relu', name = 'relu_0')(hidden)
    hidden = Dense(10, kernel_initializer=dense_kernel_init, name='dense_1' )(hidden)
    hidden = Activation('relu', name = 'relu_1')(hidden)
    hidden = Dense(3, kernel_initializer=dense_kernel_init, name='dense_2' )(hidden)
    hidden = Activation('softmax', name='output_softmax')(hidden)
    model = Model(inputs=x_in, outputs=hidden)
    
    return model

## Floating Point Model Training

In [4]:
l1_reg = 0
l2_reg = 0

## GRU Model
#model = grumodel(15, 6, 120, [50, 10], l1_reg=l1_reg, l2_reg=l2_reg)

## LSTM Model
model = lstmmodel(15, 6, 120, [50, 10], l1_reg=l1_reg, l2_reg=l2_reg)

model.compile(optimizer='adam', loss=['categorical_crossentropy'], metrics=['accuracy'])
    
#model_output = 'gru_test3/gru_weights.h5'
model_output = 'lstm_test2/lstm_weights.h5'

train = False
if train:
    history = model.fit(x_train, y_train,
            batch_size=2**10,
            epochs=150,
            validation_data=(x_val, y_val), 
            shuffle = True,
            sample_weight= w_train,
            callbacks = [
                EarlyStopping(verbose=True, patience=20, monitor='val_accuracy'),
                ModelCheckpoint(model_output, monitor='val_accuracy', verbose=True, save_best_only=True)
                ],
            verbose=True
            )
#y_keras = model.predict(x_test, batch_size=2**10)
#auc_score = roc_auc_score(y_test, y_keras)
#print("AUC score:", auc_score)

## LSTM weight

In [5]:
model = load_model("lstm_test2/lstm_weights.h5")

for layer in model.layers:
    weights = layer.get_weights()
    print(layer.name, weights)

input_6 []
lstm1 [array([[ 0.49079454, -1.1204377 , -0.05260832, ...,  0.16984597,
        -0.20320961, -0.08185341],
       [ 0.38426185, -0.808004  , -0.15241025, ...,  0.07156762,
         0.59442973,  0.05756089],
       [ 0.04227524, -0.9423642 ,  0.88680595, ...,  1.1980453 ,
         0.3316359 ,  0.3752381 ],
       [ 0.00435796, -0.10688871,  0.36524487, ...,  0.6310424 ,
        -0.24402067,  0.48337543],
       [ 0.34662625,  0.23644   ,  0.91399103, ..., -0.06033983,
         0.66045   , -0.29554167],
       [ 0.81278235,  0.18537585, -0.30137703, ..., -0.6476805 ,
        -0.16395459, -0.6595477 ]], dtype=float32), array([[-0.1512722 ,  0.16262484, -0.22720727, ..., -0.08130786,
        -0.17788623, -0.2520775 ],
       [ 0.09537277, -0.08669291,  0.30998278, ...,  0.1758767 ,
        -0.29684278, -0.07141834],
       [ 0.09804206, -0.01613149,  0.08990317, ...,  0.00711243,
        -0.04714818,  0.1428612 ],
       ...,
       [ 0.03762303, -0.08936564, -0.14454861, ...,  

## Load QLSTM Post Training Weights

In [6]:
#qgru = load_model('0int_qgru_test3/2frac_qgru_weights.h5', custom_objects={'QGRU': QGRU, 'QDense': QDense, 'quantized_bits': quantized_bits, 'QActivation': QActivation})
#qgru = 
model.load_weights('2int_qgru_test3/2frac_qgru_weights.h5')
#model_save_quantized_weights(qgru, f"ptq2int2fra_weight")

ValueError: Invalid bias shape: (2, 360)

In [ ]:
import hls4ml

config = hls4ml.utils.config_from_keras_model(model, granularity='model') #, backend='Vivado'
print("-----------------------------------")
print("Configuration")
plotting.print_dict(config)
print("-----------------------------------")
hls_model = hls4ml.converters.convert_from_keras_model(
    model, hls_config=config, backend='Vivado', output_dir='model_1/hls4ml_prj', part='xc7vx690tffg1761-2'
)